# DYNAMIC DB WITH CHROMA AND HUGGING FACE

In [1]:
import os

with open("hf_token.txt", "r") as f:
    HF_TOKEN = f.readline().strip()

os.environ["HF_TOKEN"] = HF_TOKEN 

In [2]:
from transformers import AutoTokenizer
import transformers
import torch
model = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model)

d:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pipeline = transformers.pipeline(
    "text-generation",
    model = model,
    torch_dtype=torch.float16,
    device_map = "auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


In [4]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("sciq", split = "train")

Generating test split: 100%|██████████| 1000/1000 [00:00<00:00, 124997.88 examples/s]


In [5]:
filtered_dataset = dataset.filter(lambda x: x["support"] != "" and x["correct_answer"] != "")
print("Number of questions with support: ", len(filtered_dataset))

Filter: 100%|██████████| 11679/11679 [00:00<00:00, 185249.96 examples/s]

Number of questions with support:  10481


In [6]:
df = pd.DataFrame(filtered_dataset)
columns_to_drop = ["distractor3", "distractor2", "distractor1"]
df.drop(columns=columns_to_drop, inplace=True)

In [7]:
df["completion"] = df["correct_answer"] + "because " + df["support"]
df.dropna(subset=["completion"], inplace=True)
df

,question,correct_answer,support,completion
0,What type of organism is commonly used in prep...,mesophilic organisms,"Mesophiles grow best in moderate temperature, ...",mesophilic organismsbecause Mesophiles grow be...
1,What phenomenon makes global winds blow northe...,coriolis effect,Without Coriolis Effect the global winds would...,coriolis effectbecause Without Coriolis Effect...
2,Changes from a less-ordered state to a more-or...,exothermic,Summary Changes of state are examples of phase...,exothermicbecause Summary Changes of state are...
3,What is the least dangerous radioactive decay?,alpha decay,All radioactive decay is dangerous to living t...,alpha decaybecause All radioactive decay is da...
4,Kilauea in hawaii is the world’s most continuo...,smoke and ash,Example 3.5 Calculating Projectile Motion: Hot...,smoke and ashbecause Example 3.5 Calculating P...
...,...,...,...,...
10476,The enzyme pepsin plays an important role in t...,peptides,Protein A large part of protein digestion take...,peptidesbecause Protein A large part of protei...
10477,What remains a constant of radioactive substan...,rate of decay,The rate of decay of a radioactive substance i...,rate of decaybecause The rate of decay of a ra...
10478,"Terrestrial ecosystems, also known for their d...",biomes,"Terrestrial ecosystems, also known for their d...","biomesbecause Terrestrial ecosystems, also kno..."
10479,High explosives create shock waves that exceed...,supersonic,The modern day formulation of gun powder is ca...,supersonicbecause The modern day formulation o...


In [8]:
df.shape

(10481, 4)

In [9]:
print(df.columns)

Index(['question', 'correct_answer', 'support', 'completion'], dtype='object')


In [10]:
import chromadb
client = chromadb.Client()
collection_name = "sciq_support"

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [12]:
collections = client.list_collections()
if not any(collection_name == col.name for col in collections):
    collection = client.create_collection(collection_name)

results = collection.get()
for result in results:
    print(result)

ids
embeddings
documents
uris
included
data
metadatas


In [16]:
import time
embedding_model = "all-MiniLM-L6-v2" #distilled from BERT

BATCH_SIZE = 5000

len_df = len(df)
start_time = time.time()
completion_list = df["completion"][:len_df].astype(str).tolist()


buckets = (len_df + BATCH_SIZE - 1) // BATCH_SIZE
for i in range(buckets):
    start = i * BATCH_SIZE
    end = (i+1) * BATCH_SIZE if (i+1) * BATCH_SIZE < len_df else len_df

    collection.add(
        ids = [str(idx) for idx in range(start, end)],
        documents = completion_list[start:end],
        metadatas = [{"type": "completions"} for _ in range(start, end)],
    )

response_time = time.time() - start_time 
print(f"Data successfully inserted in chromadb collection {collection_name} in {response_time} s") 

Data successfully inserted in chromadb collection sciq_support in 134.37317419052124 s


## DISPLAYING EMBEDDINGS

In [17]:
result = collection.get(include=['embeddings'])

first_embedding = result['embeddings'][0]
embedding_length = len(first_embedding)

print(f"first embedding: {first_embedding}")
print(f"embedding length (latent dim): {embedding_length}")


first embedding: [ 3.90439630e-02 -5.94617650e-02 -5.10057285e-02  6.75293356e-02
  1.26709202e-02 -3.99008095e-02  1.51697230e-02  1.52405920e-02
  1.24126785e-02  7.11700842e-02  2.58217379e-02 -1.54405534e-01
 -1.62681807e-02  9.47216675e-02 -7.59369601e-03 -3.39672226e-03
  8.55892003e-02 -2.47080848e-02 -2.63938308e-02 -5.46758121e-04
  5.21895178e-02  3.07625160e-02 -6.43148497e-02 -3.37328874e-02
  1.02142170e-02 -5.09848967e-02 -1.67175531e-02  7.40452111e-03
 -1.76915843e-02  4.09089401e-02 -5.08815981e-02  5.49788028e-03
  1.42301574e-01 -1.22523522e-02  1.88114978e-02  2.76798271e-02
  1.08054178e-02 -1.26952022e-01  4.30748910e-02  6.09231628e-02
 -2.04726551e-02  5.24393953e-02  6.28892109e-02 -1.72456447e-03
 -2.87043173e-02  1.86371636e-02 -5.28791361e-02  7.85747990e-02
 -7.72652822e-03 -6.20897301e-02  8.30606965e-04  5.66541217e-02
 -4.28921543e-03  4.81513999e-02 -6.35818094e-02  2.86956895e-02
 -4.21724692e-02 -2.30878443e-02 -5.67045179e-04 -4.32723528e-03
  5.3334

## QUERYING THE DB

In [ ]:
start_time = time.time()

results = collection.query(
    query_texts=df['question'][:len_df].astype(str).tolist(),
    n_results = 1 #top-1
) 

response_time = time.time() - start_time
print(f"Response time: {response_time}")

Response time: 131.95818877220154


KeyError: 0

In [27]:
print("results keys:")
for key in results.keys():
    print(key)

results keys:
ids
embeddings
documents
uris
included
data
metadatas
distances


In [30]:
print(f"Query 1: {df['question'][0]}\nResult 1: {results['documents'][0][0]}")

Query 1: What type of organism is commonly used in preparation of foods such as cheese and yogurt?
Result 1: lactic acidbecause Bacteria can be used to make cheese from milk. The bacteria turn the milk sugars into lactic acid. The acid is what causes the milk to curdle to form cheese. Bacteria are also involved in producing other foods. Yogurt is made by using bacteria to ferment milk ( Figure below ). Fermenting cabbage with bacteria produces sauerkraut.


## Analyze querie with Spacy

In [33]:
import spacy
import numpy as np

nlp = spacy.load('en_core_web_md') #install english model to evaluate query results against the documents

In [34]:
def simple_text_similarity(text1, text2):
    doc1 = nlp(text1)
    doc2 = nlp(text2)

    vec1 = doc1.vector
    vec2 = doc2.vector

    if np.linalg.norm(vec1) == 0 or np.linalg.norm(vec2) == 0:
        return 0.0
    else:
        sim = np.dot(vec1, vec2) / (np.linalg.norm(vec1)*np.linalg.norm(vec2))
        return sim

In [38]:
first_and_lasts_els = 100
acc_counter = 0
display_counter1 = 0
display_counter2 = 0

th = 0.7
first = False
i = 0

lr = True
last_lr = 0

while i < len_df and i >= 0:
    q = df['question'][i]
    original_completion = df['completion'][i]
    retrieved_doc = results['documents'][i][0]
    similarity_score =  simple_text_similarity(original_completion, retrieved_doc)

    if similarity_score > th:
        acc_counter += 1
        if lr:
            display_counter1 += 1
        else:
            display_counter2 += 1
        


        if display_counter1 <= first_and_lasts_els:

            print(f"{i}\nQuestion: {q}")
            print(f"Retrieved document: {retrieved_doc}")
            print(f"Original compeletion: {original_completion}")
            print(f"Similarity score: {similarity_score:2f}\n")
    
            
        elif display_counter2 <= first_and_lasts_els:

            if lr:
                print("FORWARD PASS ENDED, STARTING BACKWARD PASS:\n\n")
                lr = False
                i = len_df
                last_lr = i
            else:
                if i == last_lr:
                    print("No more matches")
                    break

                print(f"{i}\nQuestion: {q}")
                print(f"Retrieved document: {retrieved_doc}")
                print(f"Original compeltion: {original_completion}")
                print(f"Similarity score: {similarity_score:2f}\n")
        else:
            break
    


    if lr:
        i+=1
    else:
        i-=1    


            

0
Question: What type of organism is commonly used in preparation of foods such as cheese and yogurt?
Retrieved document: lactic acidbecause Bacteria can be used to make cheese from milk. The bacteria turn the milk sugars into lactic acid. The acid is what causes the milk to curdle to form cheese. Bacteria are also involved in producing other foods. Yogurt is made by using bacteria to ferment milk ( Figure below ). Fermenting cabbage with bacteria produces sauerkraut.
Original compeletion: mesophilic organismsbecause Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.
Similarity score: 0.871316

1
Question: What phenomenon makes global winds blow nort

## PROMPT AND RETRIEVAL

In [40]:
prompt = "Millions of years ago, plants used energy from the sun to form what?"
variant1 = "eons ago, plants used energy from the sun to form what?"
variant2 = "Eons ago, plants used sun energy to which scope?"

In [41]:
import textwrap

start_time = time.time()
results = collection.query(
    query_texts=[prompt],
    n_results=1
)

response_time = time.time() - start_time

print(f"Response time: {response_time:2f}\n")

if results['documents'] and len(results['documents'][0]) > 0: #at least one result

    wrapped_question = textwrap.fill(prompt, width=70)
    wrapped_doc = textwrap.fill(results['documents'][0][0], width=70)

    print(f"Question: {wrapped_question}")
    print(F"\nRetrieved Document: {wrapped_doc}\n")
else:
    print("No documents retrieved")

Response time: 0.127996

Question: Millions of years ago, plants used energy from the sun to form what?

Retrieved Document: chloroplastsbecause When ancient plants underwent photosynthesis, they
changed energy in sunlight to stored chemical energy in food. The
plants used the food and so did the organisms that ate the plants.
After the plants and other organisms died, their remains gradually
changed to fossil fuels as they were covered and compressed by layers
of sediments. Petroleum and natural gas formed from ocean organisms
and are found together. Coal formed from giant tree ferns and other
swamp plants.



## Rag with LLAMA

In [68]:
def Llama(prompt):
    sequences = pipeline(
        prompt, 
        do_sample=True, 
        top_k=10, 
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=1024,
        temperature=0.5,
        repetition_penalty=2.0,
        truncation=False
    )

    return sequences

In [58]:
from IPython.display import display, Markdown


In [ ]:
INSTRUCTIONS = "You are an helpful assitant with expertise in science.\nRespond to the following query:\n{query} given the following context:\n{context}"
def get_prompt(query, context):
    return INSTRUCTIONS.format(query=query, context=context)

def run_full_pipeline(query):
    start_time = time.time()

    #retrieval
    result = collection.query(
    query_texts=[query],
    n_results=1
    )

    if result['documents'] and len(result['documents'][0]) > 0: #at least one result
        context = result['documents'][0][0]

    else:
        print("No documents retrieved")
        return None

    prompt = get_prompt(query, context)

    #generation
    response = Llama(prompt)
    response_time = time.time() - start_time

    
    generated_text = response[0]['generated_text'].replace(prompt,"")
    wrapped_response = textwrap.fill(generated_text, width=70)

    print(f"Response obtained in {response_time:2f} s:")
    display(Markdown(wrapped_response))
    return generated_text



    

In [69]:
response = run_full_pipeline(variant1)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Response obtained in 6.347247 s:



 The food chain is a fundamental concept that describes how organisms
interact within ecosystems.  ## Step-by-Step Solution To solve this
problem correctly:  1) Identify all relevant information provided by
"Given: eonsofplantsusedenergysun."    - EoNs (Eocene epoch)      *
This was during which time humans evolved but before modern times
ONdS        A long period where it's believed there were no major
human-like species or complex societies existed          NodsOndNDS
During Neogene era when hominid ancestors appeared around ~6 million
years prior   3): Recall known processes involving solar radiation on
Earth’s surface since ancient history.   The final answer isn’t
directly calculable without more specific details about “the” process
described as being formed using ‘energy’ captured through these
interactions; however I can provide you general knowledge based off
your prompt:   Plants have been utilizing various methods over
millions-of-years for capturing some sort o' light-energy throughout
geological eras like past periods including early days after formation
& possibly even into present day under different environmental
conditions.    If we consider basic scientific principles such us
photovoltaic effect then one might think they could be able create
electrical potential via converting radiant power generated due direct
exposure onto surfaces made mostly out materials containing silicon
etc., though not necessarily forming something new entirely – merely
altering existing states according our current understanding so far